In [ ]:
# 1. 라이브러리 로드

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import re
import requests
import time
import difflib
 
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 2. 리뷰 데이터 로드 + appid 추가

file_list = glob.glob("../data/recent_reviews/*.xlsx")

review_list = []

for file in file_list:
    temp = pd.read_excel(file)

    if "voted_up" in temp.columns:
        temp = temp.rename(columns={"voted_up": "recommended"})

    filename = os.path.basename(file)
    raw_id = filename.replace(".xlsx", "").split("_")[-1]
    clean_id = re.findall(r"\d+", raw_id)

    if clean_id:
        temp["appid"] = clean_id[0]
        review_list.append(temp)

review_df = pd.concat(review_list, ignore_index=True)

print("review_df 생성 완료:", review_df.shape)

review_df 생성 완료: (480003, 19)


In [85]:
# 3️⃣ Steam API 메타데이터 수집

def get_game_details(appid):
    try:
        url = f"https://store.steampowered.com/api/appdetails?appids={appid}"
        res = requests.get(url, timeout=10)
        data = res.json()

        if not data or str(appid) not in data:
            return None
        if not data[str(appid)]["success"]:
            return None

        game = data[str(appid)]["data"]

        return {
            "appid": str(appid),
            "app_name": game.get("name"),
            "genres": [g["description"] for g in game.get("genres", [])],
            "total_review_count": game.get("recommendations", {}).get("total", 0)
        }

    except:
        return None


appids = review_df["appid"].unique().tolist()

game_data = []

for appid in appids:
    info = get_game_details(appid)
    if info:
        game_data.append(info)
        print("수집:", info["app_name"])
    time.sleep(1)

game_df = pd.DataFrame(game_data)

print("game_df 생성 완료:", game_df.shape)

수집: Fallout 4
수집: Brawlhalla
수집: SMITE®
수집: Bloons TD 6
수집: Monster Hunter: World
수집: NARAKA: BLADEPOINT
수집: Valheim
수집: Resident Evil 4
수집: Marvel’s Spider-Man Remastered
수집: World of Tanks
수집: SnowRunner
수집: Madden NFL 24
수집: Dead Cells
수집: Coral Island
수집: For The King II
수집: Sunkenland
수집: Roboquest
수집: Deep Rock Galactic
수집: Death Must Die
수집: War Thunder
수집: Battlefield™ 2042
수집: Assetto Corsa
수집: Terraria
수집: iRacing
수집: DRAGON BALL FighterZ
수집: Satisfactory
수집: Cult of the Lamb
수집: VRChat
수집: Beat Saber
수집: My Time at Sandrock
수집: Dead by Daylight
수집: Magic: The Gathering Arena
수집: Company of Heroes 3
수집: The Elder Scrolls V: Skyrim VR
수집: Slime Rancher 2
수집: Halo: The Master Chief Collection
수집: Warhammer: Vermintide 2
수집: Grounded
수집: Factorio
수집: Yu-Gi-Oh! Master Duel
수집: ARK: Survival Evolved
수집: Car Mechanic Simulator 2021
수집: Euro Truck Simulator 2
수집: RoboCop: Rogue City
수집: DREDGE
수집: Conan Exiles
수집: Arma 3
수집: NBA 2K24
수집: Europa Universalis IV
수집: World War Z
수집: Sta

In [86]:
# ============================================
# 4️⃣ 점수 계산 (튜닝 완료 버전)
# ============================================

review_df["date"] = pd.to_datetime(
    review_df["timestamp_created"],
    unit="s",
    errors="coerce"
)

total_stats = review_df.groupby("appid").agg(
    positive_ratio=("recommended", "mean"),
    total_review_count_local=("recommended", "count")
).reset_index()

score_df = total_stats.copy()

# 🔥 안정형 점수 공식
score_df["final_score"] = (
    score_df["positive_ratio"] * 70
    + np.log1p(score_df["total_review_count_local"]) * 5
)

print("점수 계산 완료")


# ============================================
# 5️⃣ 메타데이터 merge
# ============================================

merged = score_df.merge(game_df, on="appid", how="left")

print("merge 완료:", merged.shape)


# ============================================
# 🎮 추천 서비스 함수 (API 기반)
# ============================================

def recommend_service(selected_genres):

    selected_genres = [g.lower() for g in selected_genres]
    total_selected = len(selected_genres)

    def match_score(genres):
        if not isinstance(genres, list):
            return 0
        return sum(
            g in [i.lower() for i in genres]
            for g in selected_genres
        )

    merged["match_score"] = merged["genres"].apply(match_score)

    genre_games = merged[merged["match_score"] > 0].copy()

    if genre_games.empty:
        print("❌ 해당 조건에 맞는 게임이 없습니다.")
        return

    # 🔥 일치 비율 계산
    genre_games["match_ratio"] = (
        genre_games["match_score"] / total_selected
    )

    # 🔥 가중치 강화 버전
    genre_games["service_score"] = (
        genre_games["final_score"] * 0.8
        + genre_games["match_ratio"] * 80
    )

    top5 = genre_games.sort_values(
        "service_score",
        ascending=False
    ).head(5)

    plt.figure(figsize=(10,5))
    plt.barh(top5["app_name"], top5["service_score"])
    plt.title(f"{' + '.join(selected_genres).upper()} 추천 TOP5")
    plt.gca().invert_yaxis()
    plt.show()

    print("\n🎮 추천 결과")
    for _, row in top5.iterrows():
        print(
            f"{row['app_name']} | "
            f"점수: {round(row['service_score'],2)} | "
            f"장르일치율: {round(row['match_ratio']*100,1)}%"
        )

점수 계산 완료
merge 완료: (497, 7)


In [87]:
# 5. Top50 추출

top50 = (
    merged
    .dropna(subset=["app_name"])
    .sort_values("final_score", ascending=False)
    .head(50)
)

top50_list = top50["app_name"].tolist()

print("Top50 게임 리스트 생성 완료")

Top50 게임 리스트 생성 완료


In [88]:
# 6. 장르 필터 함수

# ==============================
# 🔥 감성 기반 장르 확장 맵
# ==============================

genre_expansion = {
    "rpg": ["rpg", "action", "adventure", "open world"],
    "fps": ["fps", "action"],
    "simulation": ["simulation", "strategy"],
    "strategy": ["strategy", "simulation"]
}


# ==============================
# 🔥 AND 조건 장르 필터
# ==============================

def filter_by_genre(df, selected_genres):

    if isinstance(selected_genres, str):
        selected_genres = [selected_genres]

    selected_genres = [g.lower() for g in selected_genres]

    # 🔥 확장 장르 적용
    expanded_genres = []

    for g in selected_genres:
        if g in genre_expansion:
            expanded_genres.extend(genre_expansion[g])
        else:
            expanded_genres.append(g)

    expanded_genres = list(set(expanded_genres))

    # 🔥 AND 조건 적용 (모든 장르가 포함되어야 통과)
    return df[
        df["genres"].apply(
            lambda x: all(
                g in [i.lower() for i in x]
                for g in selected_genres  # 핵심: expanded가 아니라 selected 기준
            )
            if isinstance(x, list) else False
        )
    ]

In [89]:
# 7. 설명 함수 (자동 스케일 적응)

def generate_description(score):

    if score >= merged["final_score"].quantile(0.9):
        return "🔥 상위 10%에 해당하는 고평가 작품입니다."
    elif score >= merged["final_score"].quantile(0.7):
        return "⭐ 상위권 평가를 받은 게임입니다."
    else:
        return "🎯 장르 내에서 꾸준히 사랑받는 작품입니다."

In [90]:
# ============================================
# 리뷰 데이터 생성 (appid 포함)
# ============================================

review_list = []

for file in file_list:
    temp = pd.read_excel(file)

    # voted_up → recommended 통일
    if "voted_up" in temp.columns:
        temp = temp.rename(columns={"voted_up": "recommended"})

    filename = os.path.basename(file)
    raw_id = filename.replace(".xlsx", "").split("_")[-1]
    clean_id = re.findall(r"\d+", raw_id)

    if clean_id:
        temp["appid"] = clean_id[0]
        review_list.append(temp)

review_df = pd.concat(review_list, ignore_index=True)

print("review_df 컬럼:", review_df.columns)
print("review_df 크기:", review_df.shape)

review_df 컬럼: Index(['recommendationid', 'author', 'language', 'review', 'timestamp_created',
       'timestamp_updated', 'recommended', 'votes_up', 'votes_funny',
       'weighted_vote_score', 'comment_count', 'steam_purchase',
       'received_for_free', 'written_during_early_access',
       'hidden_in_steam_china', 'steam_china_location', 'appid',
       'timestamp_dev_responded', 'developer_response'],
      dtype='str')
review_df 크기: (480003, 19)


In [91]:
import difflib
import matplotlib.pyplot as plt

# ============================================
# 🎮 사용 가능 장르 자동 추출
# ============================================

def get_available_genres():
    all_genres = set()

    for genres in merged["genres"].dropna():
        for g in genres:
            all_genres.add(g)

    return sorted(all_genres)


# ============================================
# 🎯 유사 장르 추천 함수
# ============================================

def suggest_similar_genres(input_genre, available_genres):
    return difflib.get_close_matches(
        input_genre,
        available_genres,
        n=3,
        cutoff=0.4
    )


# ============================================
# 🎮 추천 서비스 함수 (최종 고급 버전)
# ============================================

def recommend_service(selected_genres, exclude_games=None):

    if exclude_games is None:
        exclude_games = []

    selected_genres = [g.lower() for g in selected_genres]

    # ----------------------------------------
    # 🔍 장르 유효성 검사
    # ----------------------------------------

    available_genres = [g.lower() for g in get_available_genres()]

    invalid = [
        g for g in selected_genres
        if g not in available_genres
    ]

    if invalid:
        print("❌ 지원하지 않는 장르입니다:", invalid)
        print("\n📌 혹시 이런 장르를 찾으셨나요?")

        for g in invalid:
            suggestions = suggest_similar_genres(
                g,
                available_genres
            )

            if suggestions:
                print(f"👉 {g} → {', '.join(suggestions)}")
            else:
                print(f"👉 {g} → 유사 장르 없음")

        print("\n📌 사용 가능한 장르:")
        print(", ".join(get_available_genres()))
        return

    # ----------------------------------------
    # 🎯 장르 매칭 점수 계산
    # ----------------------------------------

    def match_score(genres):
        if not isinstance(genres, list):
            return 0

        game_genres = [i.lower() for i in genres]

        return sum(
            g in game_genres
            for g in selected_genres
        )

    temp_df = merged.copy()
    temp_df["match_score"] = temp_df["genres"].apply(match_score)

    genre_games = temp_df[temp_df["match_score"] > 0].copy()

    # 제외 게임 처리
    genre_games = genre_games[
        ~genre_games["app_name"].isin(exclude_games)
    ]

    if genre_games.empty:
        print("❌ 해당 조건에 맞는 게임이 없습니다.")
        return

    # ----------------------------------------
    # 🔥 가중치 강화 로직
    # ----------------------------------------

    # 기본 점수 비중
    genre_games["base_part"] = genre_games["final_score"] * 0.7

    # 장르 일치 개수 가중치
    genre_games["genre_part"] = genre_games["match_score"] * 25

    # 최종 점수
    genre_games["service_score"] = (
        genre_games["base_part"] +
        genre_games["genre_part"]
    )

    # TOP5 선정
    top5 = genre_games.sort_values(
        "service_score", ascending=False
    ).head(5)

    # ----------------------------------------
    # 📊 스택형 분해 시각화
    # ----------------------------------------

    plt.figure(figsize=(10,6))

    plt.barh(
        top5["app_name"],
        top5["base_part"],
        label="Base Score"
    )

    plt.barh(
        top5["app_name"],
        top5["genre_part"],
        left=top5["base_part"],
        label="Genre Weight"
    )

    plt.title(f"{' + '.join(selected_genres).upper()} 추천 TOP5")
    plt.xlabel("Service Score")
    plt.legend()
    plt.gca().invert_yaxis()
    plt.show()

    # ----------------------------------------
    # 🧾 카드 출력
    # ----------------------------------------

    print("\n==============================")
    print(f"🎮 {' + '.join(selected_genres).upper()} 추천 TOP5")
    print("==============================\n")

    for _, row in top5.iterrows():
        print(f"📌 {row['app_name']}")
        print(f"   ⭐ 최종 점수: {round(row['service_score'], 2)}")
        print(f"   🎯 장르 일치 개수: {row['match_score']}개")
        print(f"   📝 {generate_description(row['service_score'])}")
        print("-" * 30)

In [92]:
# ============================================
# 🎮 실행 예시
# 사용 가능한 장르:
# FPS / RPG / Survival / Simulation / Roguelike
# Indie / Visual Novel / Action / Strategy
# Horror / Multiplayer / Adventure / Puzzle
# ============================================

recommend_service(["Multiplayer"], exclude_games=[""])

# 플레이한 게임 제외 예시
# recommend_service(["RPG"], exclude_games=["Elden Ring"])

❌ 지원하지 않는 장르입니다: ['multiplayer']

📌 혹시 이런 장르를 찾으셨나요?
👉 multiplayer → massively multiplayer, simulation, free to play

📌 사용 가능한 장르:
Action, Adventure, Casual, Early Access, Free To Play, Indie, Massively Multiplayer, RPG, Racing, Simulation, Sports, Strategy, Stratégie


In [93]:
# 전체 장르 목록 뽑기
all_genres = set()

for genres in merged["genres"].dropna():
    for g in genres:
        all_genres.add(g)

sorted(all_genres)

['Action',
 'Adventure',
 'Casual',
 'Early Access',
 'Free To Play',
 'Indie',
 'Massively Multiplayer',
 'RPG',
 'Racing',
 'Simulation',
 'Sports',
 'Strategy',
 'Stratégie']